# VayuCast — train on Colab

Pull data → build the 299-column coupled feature matrix → train the multi-pollutant
LightGBM emulator (PM2.5 + PM10 + NO2, conformal-calibrated) → walk-forward backtest →
download `vayucast_model.zip`.

The ingest steps are **idempotent and resumable** — Open-Meteo rate-limits shared Colab
IPs, so if a cell stalls on 429s just **re-run that cell**; already-downloaded data is
skipped. LightGBM is CPU-bound (Colab CPU ≈ 2–3× a laptop); the T4 only helps the
optional TFT at the bottom.

In [ ]:
!git clone https://github.com/umangjzx/wcfs.git
%cd wcfs
!pip -q install -e ".[model,api,dev]"

In [ ]:
import os
# OpenAQ S3 + Open-Meteo need NO key. FIRMS_MAP_KEY enables the stubble-fire layer.
os.environ["FIRMS_MAP_KEY"] = ""      # https://firms.modaps.eosdis.nasa.gov/api/map_key/
os.environ["DATA_GOV_IN_API_KEY"] = ""
os.environ["OPENAQ_API_KEY"] = ""
START, END = "2025-10-01", "2026-02-15"

## 1 · Data — real CPCB ground truth (OpenAQ S3, keyless, ~15–25 min)

In [ ]:
!python -m ingest.openaq --start $START --end $END
!python -m ingest.cpcb --history --source cams --append --start $START --end $END

## 2 · Meteorology (ERA5 via Open-Meteo)
**Re-run this cell** if it prints 429s — it retries with long backoff, pulls month by
month, and writes incrementally, so re-running resumes where it stopped.

In [ ]:
!python -m ingest.weather --history --strict --start $START --end $END

## 3 · Fire hotspots (needs `FIRMS_MAP_KEY`; skip if you didn't set one)

In [ ]:
!python -m ingest.firms --history --start $START --end $END || echo 'FIRMS skipped'

## 4 · Features → train → backtest → zip  (~40–90 min)

In [ ]:
# skip-ingest: data is already on disk from the cells above.
# stride 3 (not 2) keeps peak RAM under Colab's ~13 GB — bump to 4 if you still OOM.
!python scripts/colab_train.py --skip-ingest --start $START --end $END --num-boost 400 --stride 3 --folds 2

In [ ]:
from google.colab import files
files.download("vayucast_model.zip")
# then locally:  unzip -o vayucast_model.zip -d models/registry/  &&  restart the API

### Quick sanity check (optional, ~15 min) — PM2.5 only
```python
!python -m models.train --fast --num-boost 300
!python -m models.backtest --fast
```

### Optional: Temporal Fusion Transformer (uses the GPU)
```python
!pip -q install "pytorch-forecasting>=1.0" lightning
# implement models/tft.py: TimeSeriesDataSet over data/processed/features.parquet with
# known/observed/static splits mirroring models/dataset.py, then fit on the T4.
```